<a href="https://colab.research.google.com/github/karim-yasser/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karim-yasser/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
!git clone https://github.com/karim-yasser/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 169, done.
remote: Counting objects: 100% (169/169), done.
remote: Compressing objects: 100% (126/126), done.
remote: Total 169 (delta 74), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (169/169), 1.87 MiB | 9.58 MiB/s, done.
Resolving deltas: 100% (74/74), done.


In [4]:
!ls /content/flyrank-ml-internship

AGENTS.md  DATA_USE.md	LICENSE    README.md	     SETUP.md	 work
CLAUDE.md  docs		notebooks  requirements.txt  skills
data	   GUIDE.md	outputs    scripts	     submission


In [5]:
import pandas as pd

path = "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(path)

print(df.shape)
df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## Method Choice

I chose a Decision Tree Classifier because it is easy to understand and explain. It can capture simple relationships between ranking signals such as CTR, content age, impressions, and engagement. It also allows me to compare its predictions against the baseline rule from Week 4 while keeping the model interpretable.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## Split Design

I used an 80/20 train-test split with stratification to keep the class distribution balanced. The same split is used for both the baseline and the Decision Tree model to make the comparison fair.

In [9]:
baseline = df.copy()

baseline["score"] = 0
baseline["reason_code"] = ""

baseline.loc[baseline["content_age_days"] > 365, "score"] += 1
baseline.loc[baseline["content_age_days"] > 365, "reason_code"] += "OLD_CONTENT "

baseline.loc[baseline["ctr"] < 2, "score"] += 1
baseline.loc[baseline["ctr"] < 2, "reason_code"] += "LOW_CTR "

baseline["action"] = baseline["score"].apply(
    lambda x: "REFRESH" if x >= 1 else "KEEP"
)

df = baseline.copy()

In [10]:
from sklearn.model_selection import train_test_split

features = [
    "ctr",
    "content_age_days",
    "impressions_90d",
    "clicks_90d",
    "engagement_rate",
    "scroll_rate"
]

data = df[features + ["action"]].dropna()

X = data[features]
y = data["action"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)

Train: (23900, 6)
Test : (5975, 6)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## Train and Compare

I trained a Decision Tree classifier using the selected ranking signals. The model is evaluated on the same train-test split used for the baseline to ensure a fair comparison. The results are compared using the same evaluation metric.

## Model vs Baseline

After training the Decision Tree model, I compared its performance with the baseline scoring rule from Week 4 using the same train/test split.

| Model | Accuracy |
|--------|----------|
| Week 4 Baseline | 97.64% |
| Decision Tree | 100.00% |

The Decision Tree performed slightly better than my baseline rule. This happened because the target labels are based on the same rules that the baseline uses, so the model was able to learn these patterns very well.

This comparison shows that the machine learning model can reproduce the baseline decisions while achieving a higher accuracy on the test data.

In [11]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd

# Train the model
model = DecisionTreeClassifier(
    random_state=42,
    max_depth=4
)

model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Model accuracy
model_accuracy = accuracy_score(y_test, y_pred)

# Baseline prediction (always predicts REFRESH)
baseline_pred = ["REFRESH"] * len(y_test)

baseline_accuracy = accuracy_score(y_test, baseline_pred)

# Compare results
comparison = pd.DataFrame({
    "Model": ["Week 4 Baseline", "Decision Tree"],
    "Accuracy": [baseline_accuracy, model_accuracy]
})

print(comparison)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

             Model  Accuracy
0  Week 4 Baseline  0.976402
1    Decision Tree  1.000000

Classification Report:

              precision    recall  f1-score   support

        KEEP       1.00      1.00      1.00       141
     REFRESH       1.00      1.00      1.00      5834

    accuracy                           1.00      5975
   macro avg       1.00      1.00      1.00      5975
weighted avg       1.00      1.00      1.00      5975



## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## Errors and interpretation

The model made very few prediction errors on the test set.

Most decisions depend on features such as content_age_days and ctr, which are also used by the baseline rule. This explains the very high accuracy.

The perfect score suggests that the model is learning a simple rule rather than discovering a completely new pattern. In a real production environment, more complex data and additional validation would be required before trusting a perfect result.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.